<a href="https://colab.research.google.com/github/seankim8724/AIFFEL_quest_eng/blob/main/NLP/NLP03/Untitled8_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Install python3-dev for building Python packages with C extensions
!apt-get update
!apt-get install -y python3-dev

# Re-install mecab-python using pip, as it might now find the necessary build tools
%pip install mecab-python

# Original command from the cell, if needed, though it seems unrelated to the error
# !mkdir -p ~/work/transformer_chatbot/data/spa-eng


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:13 https://r2u.stat.ill

In [ ]:
!python3 -m pip install --upgrade pip
!python3 -m pip install konlpy # Python 3.x
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh) # MeCab 설치하기

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 119.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [konlpy]
Install mecab-ko
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1381k  100 1381k    0     0   505k      0  0:00:02  0:00:02 --:--:--  963k
mecab-0.996-ko-0.9.2/
mecab-0.996-ko-0.9.2/example/
mecab-0.996-ko-0.9.2/example/example.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.c
mecab-0.996-ko-0.9.2/example/example.c
mecab-0.996-ko-0.9.2/example/thread_test.cpp
mecab-0.996-ko-0.9.2/mecab-config.in
mec

In [ ]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.10.0+cu128


In [ ]:
import urllib.request
import zipfile

zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

print("슝=3")

슝=3


In [ ]:
extracted_folder = "./spa-eng"
file_path = os.path.join(extracted_folder, "spa.txt")

with open(file_path, "r") as f:
    spa_eng_sentences = f.read().splitlines()

spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(spa_eng_sentences)
print("Example:", total_sentence_count)

for sen in spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Example: 118964
>> We must have something to live for.	Tenemos que tener alguna razón para vivir.
>> Some people seem to always want to have the last word.	Algunas personas parecen querer tener siempre la última palabra.
>> Tom burst into tears.	Tom rompió a llorar.
>> I gave him a few books.	Le di algunos libros.
>> What are you trying to do?	¿Qué tratas de hacer?


In [ ]:
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

### 🔧 튜터봇 수정사항: Step 3. 데이터 토큰화 (누락된 셀 복구)\n
이전 과정에서 `build_corpus` 함수 및 토큰화 로직이 누락되어 `que_corpus`가 정의되지 않는 에러가 발생할 수 있습니다. 이를 해결하기 위해 토큰화 코드를 다시 추가했습니다.

In [ ]:
from konlpy.tag import Mecab

mecab = Mecab()

def build_corpus(src_data, tgt_data, tokenize_fn, max_len=40):
    que_corpus = []
    ans_corpus = []

    seen_src = set()
    seen_tgt = set()

    for src, tgt in zip(src_data, tgt_data):
        clean_src = preprocess_sentence(src)
        clean_tgt = preprocess_sentence(tgt)

        if clean_src in seen_src or clean_tgt in seen_tgt:
            continue

        seen_src.add(clean_src)
        seen_tgt.add(clean_tgt)

        tokenized_src = tokenize_fn(clean_src)
        tokenized_tgt = tokenize_fn(clean_tgt)

        if len(tokenized_src) <= max_len and len(tokenized_tgt) <= max_len:
            que_corpus.append(tokenized_src)
            ans_corpus.append(tokenized_tgt)

    return que_corpus, ans_corpus

que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs, max_len=40)
print(f"구축된 que_corpus 크기: {len(que_corpus)}")
print(f"구축된 ans_corpus 크기: {len(ans_corpus)}")

### 🔧 튜터봇 수정사항: Step 4. Augmentation (Gensim 버전 호환성 패치)\n
Kyubyong님의 `ko.bin` 파일은 이전 버전(Gensim 3.x)에서 학습되어 현재 Colab의 최신 버전(4.x)과 충돌합니다. `Could not load model: 'Word2Vec' object has no attribute 'wv'` 에러를 막기 위해 아래 셀에서 **Gensim을 3.8.3 버전으로 다운그레이드**합니다.\n
\n
**🚨 매우 중요 🚨**\n
아래 `!pip install gensim==3.8.3` 셀을 실행하신 후, **반드시 상단 메뉴에서 런타임(커널)을 다시 시작**해 주셔야 합니다! (코랩의 경우: `런타임` -> `세션 다시 시작`)

In [ ]:
!pip install gensim==3.8.3

In [ ]:
import gensim
import random
from tqdm import tqdm
import os
import zipfile

# 1. Word2Vec 모델 경로 설정
w2v_path = '/content/ko.bin'
zip_path = '/content/ko.zip'

# 파일 관리
if os.path.exists(w2v_path) and os.path.getsize(w2v_path) < 1000000:
    os.remove(w2v_path)

if not os.path.exists(w2v_path):
    if not os.path.exists(zip_path):
        !pip install -q gdown
        !gdown --id 1G0e_aEaB2-J03V0n3l3k6Q -O /content/ko.zip
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
    except Exception as e: print(f"Extraction error: {e}")

try:
    print(f"Loading vectors from {w2v_path} with gensim {gensim.__version__}...")
    # Gensim 3.8.3 환경에서 정상 로드
    wv_model = gensim.models.Word2Vec.load(w2v_path)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Could not load model: {e}")
    wv_model = None

def lexical_sub(sentence, model):
    if model is None: return None
    res = []
    try:
        toks = sentence if isinstance(sentence, list) else sentence.split()

        # Gensim 3.x 버전에 맞는 단어 확인
        valid_toks = [t for t in toks if t in model.wv.vocab]
        if not valid_toks: return None

        _from = random.choice(valid_toks)
        _to = model.wv.most_similar(_from)[0][0]

        for tok in toks:
            res.append(_to if tok == _from else tok)
        return res
    except Exception: return None

# Augmentation logic
aug_que_corpus = []
aug_ans_corpus = []

if wv_model is not None:
    aug_que_corpus.extend(que_corpus)
    aug_ans_corpus.extend(ans_corpus)

    for q, a in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus), desc="Augmenting"):
        q_sub = lexical_sub(q, wv_model)
        aug_que_corpus.append(q_sub if q_sub else q)
        aug_ans_corpus.append(a)

        a_sub = lexical_sub(a, wv_model)
        aug_que_corpus.append(q)
        aug_ans_corpus.append(a_sub if a_sub else a)

    print(f"Final Augmented Dataset Size: {len(aug_que_corpus)}")
else:
    print(f"Word2Vec model is unavailable. Proceeding with original dataset (Size: {len(que_corpus)}).")
    aug_que_corpus = list(que_corpus)
    aug_ans_corpus = list(ans_corpus)

In [ ]:
spa_eng_sentences = list(map(preprocess_sentence, spa_eng_sentences))

print('슝=3')

슝=3


In [ ]:
test_sentence_count = total_sentence_count // 200
print("Test Size: ", test_sentence_count)
print("\n")

train_spa_eng_sentences = spa_eng_sentences[:-test_sentence_count]
test_spa_eng_sentences = spa_eng_sentences[-test_sentence_count:]
print("Train Example:", len(train_spa_eng_sentences))
for sen in train_spa_eng_sentences[0:100][::20]:
    print(">>", sen)
print("\n")
print("Test Example:", len(test_spa_eng_sentences))
for sen in test_spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Test Size:  594


Train Example: 118370
>> we must have something to live for.	tenemos que tener alguna razón para vivir.
>> some people seem to always want to have the last word.	algunas personas parecen querer tener siempre la última palabra.
>> tom burst into tears.	tom rompió a llorar.
>> i gave him a few books.	le di algunos libros.
>> what are you trying to do?	¿qué tratas de hacer?


Test Example: 594
>> smell this.	huelan esto.
>> tom is the taller of the two boys.	tom es el más alto de los dos chicos.
>> this paper is rough.	este papel es áspero.
>> if it's at all possible, i'd like you to take part in the next meeting.	de ser posible, me gustaría que participaras en la próxima reunión.
>> i've saved the best for last.	he dejado lo mejor para el final.


In [ ]:
def split_spa_eng_sentences(spa_eng_sentences):
    spa_sentences = []
    eng_sentences = []
    for spa_eng_sentence in tqdm(spa_eng_sentences):
        eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        spa_sentences.append(spa_sentence)
        eng_sentences.append(eng_sentence)
    return eng_sentences, spa_sentences

print('슝=3')

슝=3


In [ ]:
train_eng_sentences, train_spa_sentences = split_spa_eng_sentences(train_spa_eng_sentences)
print(len(train_eng_sentences))
print(train_eng_sentences[0])
print('\n')
print(len(train_spa_sentences))
print(train_spa_sentences[0])

  0%|          | 0/118370 [00:00<?, ?it/s]

118370
we must have something to live for.


118370
tenemos que tener alguna razón para vivir.


In [ ]:
test_eng_sentences, test_spa_sentences = split_spa_eng_sentences(test_spa_eng_sentences)
print(len(test_eng_sentences))
print(test_eng_sentences[0])
print('\n')
print(len(test_spa_sentences))
print(test_spa_sentences[0])

  0%|          | 0/594 [00:00<?, ?it/s]

594
smell this.


594
huelan esto.


## 토큰화

In [ ]:
def generate_tokenizer(corpus,
                       vocab_size,
                       lang="spa-eng",
                       pad_id=0,   # pad token의 일련번호
                       bos_id=1,  # 문장의 시작을 의미하는 bos token(<s>)의 일련번호
                       eos_id=2,  # 문장의 끝을 의미하는 eos token(</s>)의 일련번호
                       unk_id=3):   # unk token의 일련번호
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=./%s --model_prefix=%s --vocab_size=%d'\
        % (file, model, vocab_size) + \
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'\
        % (pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer

print("슝=3")

슝=3


In [ ]:
VOCAB_SIZE = 20000
tokenizer = generate_tokenizer(train_eng_sentences + train_spa_sentences, VOCAB_SIZE, 'spa-eng')
tokenizer.set_encode_extra_options("bos:eos")  # 문장 양 끝에 <s> , </s> 추가

True

In [ ]:
def make_corpus(sentences, tokenizer):
    corpus = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.encode_as_ids(sentence)
        corpus.append(tokens)
    return corpus

print('슝=3')

슝=3


In [ ]:
eng_corpus = make_corpus(train_eng_sentences, tokenizer)
spa_corpus = make_corpus(train_spa_sentences, tokenizer)

  0%|          | 0/118370 [00:00<?, ?it/s]

  0%|          | 0/118370 [00:00<?, ?it/s]

In [ ]:
print(train_eng_sentences[0])
print(eng_corpus[0])
print('\n')
print(train_spa_sentences[0])
print(spa_corpus[0])

we must have something to live for.
[1, 46, 236, 39, 189, 10, 346, 44, 0, 2]


tenemos que tener alguna razón para vivir.
[1, 383, 15, 557, 360, 815, 65, 680, 0, 2]


In [ ]:
MAX_LEN = 50

def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    """
    sequences: list of list (각 문장별 토큰 ID 리스트)
    max_len: 고정할 최대 시퀀스 길이
    pad_value: 패딩에 사용할 값 (일반적으로 0)
    """
    padded_sequences = []

    for seq in sequences:
        # 초과 길이는 자르고
        if len(seq) > max_len:
            seq = seq[:max_len]
        # 부족한 길이는 pad_value로 채우기
        else:
            seq = seq + [pad_value] * (max_len - len(seq))

        padded_sequences.append(seq)

    # 최종적으로 torch.Tensor로 변환 (shape: [batch_size, max_len])
    return torch.tensor(padded_sequences, dtype=torch.long)

enc_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)
dec_ndarray = pad_sequences_custom(spa_corpus, max_len=MAX_LEN, pad_value=0)

print(enc_ndarray.shape)  # 예) [batch_size, 50]
print(dec_ndarray.shape)  # 예) [batch_size, 50]
print("슝=3")

torch.Size([118370, 50])
torch.Size([118370, 50])
슝=3


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = TensorDataset(enc_ndarray, dec_ndarray)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print("슝=3")

슝=3


## 번역 모델 만들기

transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)

# Positional Encoding

In [ ]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
print("슝=3")

슝=3


# 마스크 생성

In [ ]:
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    enc_mask = generate_padding_mask(src)
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    dec_enc_mask = generate_padding_mask(src)

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

슝=3


# Multi-head Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

print("슝=3")

슝=3


# Position-wise Feed Forward Network

In [ ]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        out = self.fc2(out)          # 두 번째 Dense
        return out

print("슝=3")

슝=3


# Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # nn.LayerNorm은 마지막 차원(d_model)을 기준으로 정규화
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention 단계
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual  # residual connection

        # Position-Wise Feed Forward 단계
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual  # residual connection

        return out, enc_attn

print("슝=3")

슝=3


# Decoder Layer

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn

print("슝=3")

슝=3


# Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask):
        out = x; enc_attns = []
        for layer in self.enc_layers:
            out, enc_attn = layer(out, mask)
            enc_attns.append(enc_attn)
        out = self.final_norm(out)
        return out, enc_attns

# 사용 예시: Encoder 인스턴스 생성 후 forward 호출
# encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
# out, enc_attns = encoder(x, mask)
print("슝=3")

슝=3


# Decoder

In [ ]:
import torch.nn as nn

class Decoder (nn .Module):
  def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
    super().__init__()
    self.n_layers = n_layers
    self.dec_layers = nn.ModuleList(
      [DecoderLayer (d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
    self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

  def forward(self, x, enc_out, dec_enc_mask, padding_mask):
    out = x; dec_attns = []; dec_enc_attns = []
    for layer in self.dec_layers:
      out, dec_attn, dec_enc_attn = layer(out, enc_out, dec_enc_mask, padding_mask)
      dec_attns.append(dec_attn)
      dec_enc_attns.append(dec_enc_attn)
    out = self.final_norm(out) #~ stack 끝나고 정규화
    return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


# Transformer 전체 모델 조립

In [ ]:
import math

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))

        self.do = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


# 모델 인스턴스 생성

In [ ]:
# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)

transformer = transformer.to(device)

d_model = 512

print("슝=3")

슝=3


# Learning Rate Scheduler

In [ ]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60): # 4000
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

print("슝=3")

슝=3


#Learning Rate & Optimizer

In [ ]:
# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)

print("슝=3")

슝=3


#Loss Function 정의

In [ ]:
def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

print("슝=3")

슝=3


# Train Step 정의

In [ ]:
def train_step(src, tgt, model, optimizer):
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


# 훈련을 시키자!

In [ ]:
%%time

EPOCHS = 3

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    for batch, (src, tgt) in enumerate(train_dataloader):
        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)

        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch+1}, Loss: {total_loss / dataset_count:.4f}")

  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 1, Loss: 42.1786


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 2, Loss: 14.3196


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 3, Loss: 10.9768
CPU times: user 14min 32s, sys: 3.04 s, total: 14min 35s
Wall time: 14min 49s


# 번역 성능 측정하기

## NLTK를 활용한 BLEU Score

In [ ]:
# 아래 두 문장을 바꿔가며 테스트 해보세요
reference = "많 은 자연어 처리 연구자 들 이 트랜스포머 를 선호 한다".split()
candidate = "적 은 자연어 학 개발자 들 가 트랜스포머 을 선호 한다 요".split()

print("원문:", reference)
print("번역문:", candidate)
print("BLEU Score:", sentence_bleu([reference], candidate))

원문: ['많', '은', '자연어', '처리', '연구자', '들', '이', '트랜스포머', '를', '선호', '한다']
번역문: ['적', '은', '자연어', '학', '개발자', '들', '가', '트랜스포머', '을', '선호', '한다', '요']
BLEU Score: 8.190757052088229e-155


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [ ]:
print("1-gram:", sentence_bleu([reference], candidate, weights=[1, 0, 0, 0]))
print("2-gram:", sentence_bleu([reference], candidate, weights=[0, 1, 0, 0]))
print("3-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 1, 0]))
print("4-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 0, 1]))

1-gram: 0.5
2-gram: 0.18181818181818182
3-gram: 2.2250738585072626e-308
4-gram: 2.2250738585072626e-308


## SmoothingFunction() 으로 BLEU Score 보정하기

In [ ]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                         candidate,
                         weights=weights,
                         smoothing_function=SmoothingFunction().method1)  # smoothing_function 적용

print("BLEU-1:", calculate_bleu(reference, candidate, weights=[1, 0, 0, 0]))
print("BLEU-2:", calculate_bleu(reference, candidate, weights=[0, 1, 0, 0]))
print("BLEU-3:", calculate_bleu(reference, candidate, weights=[0, 0, 1, 0]))
print("BLEU-4:", calculate_bleu(reference, candidate, weights=[0, 0, 0, 1]))

print("\nBLEU-Total:", calculate_bleu(reference, candidate))

BLEU-1: 0.5
BLEU-2: 0.18181818181818182
BLEU-3: 0.010000000000000004
BLEU-4: 0.011111111111111112

BLEU-Total: 0.05637560315259291


## 트랜스포머 모델의 번역 성능 알아보기

In [ ]:
import torch
import torch.nn.functional as F

def translate(tokens, model, src_tokenizer, tgt_tokenizer):
    # tokens: 입력 토큰 리스트
    # MAX_LEN: 최대 길이 (전역 변수 혹은 상수)
    # device: 모델과 데이터가 위치한 디바이스

    # tokens 길이가 MAX_LEN보다 크면 자르고, 작으면 0으로 패딩
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN]
    else:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    # 배치 차원을 추가하여 텐서로 변환 (shape: [1, MAX_LEN])
    padded_tokens = torch.tensor([tokens], dtype=torch.long, device=device)

    ids = []
    # 디코더의 첫 입력은 BOS 토큰 (배치 차원 추가)
    output = torch.tensor([[tgt_tokenizer.bos_id()]], dtype=torch.long, device=device)

    for i in range(MAX_LEN):
        # generate_masks는 padded_tokens와 현재 output으로부터 마스크들을 생성합니다.
        enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(padded_tokens, output)

        # 모델 예측: predictions shape: [batch, seq_len, num_classes]
        predictions, _, _, _ = model(padded_tokens, output, enc_padding_mask, combined_mask, dec_padding_mask)

        # 마지막 시퀀스 위치의 예측값을 소프트맥스 후 argmax로 선택
        predicted_id = predictions[0, -1].softmax(dim=-1).argmax(dim=-1).item()

        # EOS 토큰에 도달하면 현재까지의 예측 토큰 ids를 디코딩 후 반환
        if tgt_tokenizer.eos_id() == predicted_id:
            result = tgt_tokenizer.decode_ids(ids)
            return result

        ids.append(predicted_id)
        # 현재 output에 새로운 예측 토큰을 연결 (dim=1)
        new_token = torch.tensor([[predicted_id]], dtype=torch.long, device=device)
        output = torch.cat([output, new_token], dim=1)

    result = tgt_tokenizer.decode_ids(ids)
    return result

print("슝=3")

슝=3


In [ ]:
def eval_bleu_single(model, src_sentence, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    src_tokens = src_tokenizer.encode_as_ids(src_sentence)
    tgt_tokens = tgt_tokenizer.encode_as_ids(tgt_sentence)

    if (len(src_tokens) > MAX_LEN): return None
    if (len(tgt_tokens) > MAX_LEN): return None

    reference = tgt_sentence.split()
    candidate = translate(src_tokens, model, src_tokenizer, tgt_tokenizer).split()

    score = sentence_bleu([reference], candidate,
                          smoothing_function=SmoothingFunction().method1)

    if verbose:
        print("Source Sentence: ", src_sentence)
        print("Model Prediction: ", candidate)
        print("Real: ", reference)
        print("Score: %lf\n" % score)

    return score

print('슝=3')

슝=3


In [ ]:
# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 5

eval_bleu_single(transformer,
                 test_eng_sentences[test_idx],
                 test_spa_sentences[test_idx],
                 tokenizer,
                 tokenizer)

Source Sentence:  you're something else.
Model Prediction:  ['tú', '⁇', 'parece', 'australia', 'a', 'tu', 'trabajo', 'mejores,', 'cuenta', 'al', 'que', 'decirle', 'a', 'eso', 'por', 'la', 'puntual', 'últimas', 'los', 'blanco', 'eso', 'a', 'duro', 'demasiado?', 'que', 'leído', 'en', 'por', 'ella', 'toca,', 'le', 'gusta', 'de', 'la', 'libros,', 'que', 'dónde', 'acostumbrado', 'a', 'un', 'día', 'que']
Real:  ['eres', 'la', 'leche.']
Score: 0.004392



0.004392487796991638

In [ ]:
def eval_bleu(model, src_sentences, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    total_score = 0.0
    sample_size = len(src_sentences)

    for idx in tqdm(range(sample_size)):
        score = eval_bleu_single(model, src_sentences[idx], tgt_sentence[idx], src_tokenizer, tgt_tokenizer, verbose)
        if not score: continue

        total_score += score

    print("Num of Sample:", sample_size)
    print("Total Score:", total_score / sample_size)

print("슝=3")

슝=3


In [ ]:
eval_bleu(transformer, test_eng_sentences, test_spa_sentences, tokenizer, tokenizer, verbose=False)

  0%|          | 0/594 [00:00<?, ?it/s]

Num of Sample: 594
Total Score: 0.0040105094794919294


## Beam Search Decoder

In [ ]:
def beam_search_decoder(prob, beam_size):
    sequences = [[[], 1.0]]  # 생성된 문장과 점수를 저장

    for tok in prob:
        all_candidates = []

        for seq, score in sequences:
            for idx, p in enumerate(tok): # 각 단어의 확률을 총점에 누적 곱
                candidate = [seq + [idx], score * -math.log(-(p-1))]
                all_candidates.append(candidate)

        ordered = sorted(all_candidates,
                         key=lambda tup:tup[1],
                         reverse=True) # 총점 순 정렬
        sequences = ordered[:beam_size] # Beam Size에 해당하는 문장만 저장

    return sequences

print("슝=3")

슝=3


In [ ]:
vocab = {
    0: "<pad>",
    1: "까요?",
    2: "커피",
    3: "마셔",
    4: "가져",
    5: "될",
    6: "를",
    7: "한",
    8: "잔",
    9: "도",
}

prob_seq = [[0.01, 0.01, 0.60, 0.32, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.75, 0.01, 0.01, 0.17],
            [0.01, 0.01, 0.01, 0.35, 0.48, 0.10, 0.01, 0.01, 0.01, 0.01],
            [0.24, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.68],
            [0.01, 0.01, 0.12, 0.01, 0.01, 0.80, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.81, 0.01, 0.01, 0.01, 0.01, 0.11, 0.01, 0.01, 0.01],
            [0.70, 0.22, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]]

prob_seq = np.array(prob_seq)
beam_size = 3

result = beam_search_decoder(prob_seq, beam_size)

for seq, score in result:
    sentence = ""

    for word in seq:
        sentence += vocab[word] + " "

    print(sentence, "// Score: %.4f" % score)

커피 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 42.5243
커피 를 마셔 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 28.0135
마셔 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 17.8983


## Beam Search Decoder 작성 및 평가하기

In [ ]:
import torch
import torch.nn.functional as F

def calc_prob(src_ids, tgt_ids, model):
    # 마스크 생성 (PyTorch 버전)
    enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(src_ids, tgt_ids)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src_ids,
        tgt_ids,
        enc_padding_mask,
        combined_mask,
        dec_padding_mask
    )

    # 마지막 차원에 대해 softmax 적용하여 확률값 계산
    return F.softmax(predictions, dim=-1)

print("슝=3")

슝=3


In [ ]:
import numpy as np
import torch

def beam_search_decoder(sentence,
                        src_len,
                        tgt_len,
                        model,
                        src_tokenizer,
                        tgt_tokenizer,
                        beam_size):
    # 입력 문장을 토큰화
    tokens = src_tokenizer.encode_as_ids(sentence)

    # src_in: [1, src_len] 크기의 텐서로 padding (0: 패딩 토큰)
    padded = np.zeros((1, src_len), dtype=np.int64)
    padded[0, :len(tokens)] = tokens
    src_in = torch.tensor(padded, dtype=torch.long, device=device)

    # beam search용 캐시 배열들
    pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
    pred_tmp = np.zeros((beam_size, tgt_len), dtype=np.int64)

    eos_flag = np.zeros((beam_size,), dtype=np.int64)  # EOS를 만난 branch 표시 (EOS: -1)
    scores = np.ones((beam_size,), dtype=np.float32)     # 각 branch의 score (확률 곱)

    # 디코더 첫 입력은 BOS 토큰
    pred_tmp[:, 0] = tgt_tokenizer.bos_id()

    # 초기 디코더 입력 (branch 0의 첫 토큰) -> shape: [1, 1]
    dec_in = torch.tensor(pred_tmp[0, :1], dtype=torch.long, device=device).unsqueeze(0)
    # calc_prob()는 softmax를 적용한 확률 텐서를 반환함
    prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

    # seq_pos: 디코더 시퀀스 위치
    for seq_pos in range(1, tgt_len):
        score_cache = np.ones((beam_size * beam_size,), dtype=np.float32)

        # 각 beam branch에 대해 캐시 초기화
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
            pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = pred_tmp[branch_idx, :seq_pos]

        # 각 beam branch에 대해 후보 확률 계산 및 캐시 업데이트
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            if seq_pos != 1:
                # 해당 branch의 현재까지의 시퀀스를 디코더 입력으로 변환
                dec_in_np = pred_cache[branch_idx, :seq_pos]
                dec_in = torch.tensor(dec_in_np, dtype=torch.long, device=device).unsqueeze(0)
                prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

            # 각 branch 내에서 beam_size만큼의 후보 토큰을 선택
            for beam_idx in range(beam_size):
                max_idx = np.argmax(prob)
                # 후보 branch의 score 업데이트 (곱셈으로 누적)
                score_cache[cache_pos + beam_idx] *= prob[max_idx]
                pred_cache[cache_pos + beam_idx, seq_pos] = max_idx
                # 이미 선택된 토큰은 다시 선택되지 않도록 -1로 마킹
                prob[max_idx] = -1

        # 각 beam branch에서 최고 score를 가진 후보를 선택
        for beam_idx in range(beam_size):
            if eos_flag[beam_idx] == -1:
                continue
            max_idx = np.argmax(score_cache)
            prediction = pred_cache[max_idx, :seq_pos+1].copy()
            pred_tmp[beam_idx, :seq_pos+1] = prediction
            scores[beam_idx] = score_cache[max_idx]
            score_cache[max_idx] = -1  # 해당 후보 제거

            # 만약 EOS 토큰이면 해당 branch는 종료 표시 (-1)
            if prediction[-1] == tgt_tokenizer.eos_id():
                eos_flag[beam_idx] = -1

    # 각 branch의 예측 시퀀스에서 EOS 토큰 이전까지만 추출하여 결과 반환
    pred = []
    for long_pred in pred_tmp:
        eos_token = tgt_tokenizer.eos_id()
        # EOS 토큰이 없는 경우, 전체 시퀀스를 사용하도록 처리할 수 있음
        try:
            eos_idx = list(long_pred).index(eos_token)
        except ValueError:
            eos_idx = tgt_len - 1
        short_pred = long_pred[:eos_idx+1]
        pred.append(short_pred.tolist())

    return pred

print("슝=3")

슝=3


In [ ]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                            candidate,
                            weights=weights,
                            smoothing_function=SmoothingFunction().method1)

print('슝=3')

슝=3


In [ ]:
import pandas as pd
import urllib.request
import os

# Step 1. 데이터 다운로드
csv_url = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
csv_filename = "ChatbotData.csv"

# Check if the file already exists, if not, download it
if not os.path.exists(csv_filename):
    print(f"Downloading {csv_filename}...")
    urllib.request.urlretrieve(csv_url, csv_filename)
    print(f"{csv_filename} downloaded.")
else:
    print(f"{csv_filename} already exists.")

# Read the CSV file
data = pd.read_csv(csv_filename)

# Extract questions and answers
questions = data['Q'].tolist()
answers = data['A'].tolist()

print(f"Loaded {len(questions)} questions and {len(answers)} answers.")
print("First 5 questions:")
for i in range(5):
    print(f"Q: {questions[i]}")
    print(f"A: {answers[i]}")


ChatbotData.csv already exists.
Loaded 11823 questions and 11823 answers.
First 5 questions:
Q: 12시 땡!
A: 하루가 또 가네요.
Q: 1지망 학교 떨어졌어
A: 위로해 드립니다.
Q: 3박4일 놀러가고 싶다
A: 여행은 언제나 좋죠.
Q: 3박4일 정도 놀러가고 싶다
A: 여행은 언제나 좋죠.
Q: PPL 심하네
A: 눈살이 찌푸려지죠.


In [ ]:
test_idx = 5

# The beam_search_decoder returns a list of candidate token ID sequences.
# We assume the first element of the returned list is the best candidate.
ids = beam_search_decoder(
    test_eng_sentences[test_idx],
    MAX_LEN,
    MAX_LEN,
    transformer,
    tokenizer,
    tokenizer,
    beam_size=5
)

# Take the first (best) candidate sequence of token IDs.
best_candidate_ids = ids[0]

# Decode the token IDs into a sentence string and then split it into words for BLEU calculation.
candidate_sentence_tokens = tokenizer.decode_ids(best_candidate_ids).split()

# The reference Spanish sentence needs to be split into words for BLEU calculation.
reference_sentence_tokens = test_spa_sentences[test_idx].split()

# Use the defined calculate_bleu function
bleu = calculate_bleu(reference_sentence_tokens, candidate_sentence_tokens)
print(bleu)

0.004503778123700044


# 데이터 부풀리기

In [ ]:
!pip install gensim
import gensim.downloader as api

wq = api.load('glove-wiki-gigaword-300')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 144.6 MB/s  0:00:00
[==================================================] 100.0% 376.1/376.1MB downloaded


In [ ]:
wq.most_similar("banana")

[('bananas', 0.6691170930862427),
 ('mango', 0.5804104208946228),
 ('pineapple', 0.5492372512817383),
 ('coconut', 0.5462778806686401),
 ('papaya', 0.541056752204895),
 ('fruit', 0.52181077003479),
 ('growers', 0.4877638816833496),
 ('nut', 0.48399588465690613),
 ('peanut', 0.48062023520469666),
 ('potato', 0.48061180114746094)]

In [ ]:
sample_sentence = "you know ? all you need is attention ."
sample_tokens = sample_sentence.split()

selected_tok = random.choice(sample_tokens)

result = ""
for tok in sample_tokens:
    if tok is selected_tok:
        result += wq.most_similar(tok)[0][0] + " "

    else:
        result += tok + " "

print("From:", sample_sentence)
print("To:", result)

From: you know ? all you need is attention .
To: you know ? those you need is attention . 


# Lexical Substitution 구현하기

In [ ]:
# Q. Lexical Substitution 을 구현해봅시다.
def lexical_sub(sentence, wv):
    # 문장을 토큰화
    tokens = sentence.split()

    # 유효한 단어 필터링 (임베딩에 존재하는 단어만 고려)
    valid_tokens = [tok for tok in tokens if tok in wq]

    # 대체할 단어 선택 (임베딩 내 존재하는 단어 중 하나)
    if not valid_tokens:
        return sentence  # 모든 단어가 임베딩 내에 없으면 원래 문장 반환

    selected_tok = random.choice(valid_tokens)

    # 가장 유사한 단어 찾기
    similar_word = wv.most_similar(selected_tok)[0][0]

    # 변환된 문장 생성
    new_sentence = " ".join([similar_word if tok == selected_tok else tok for tok in tokens])

    return new_sentence

In [ ]:
new_corpus = []

for old_src in tqdm(test_eng_sentences):
    new_src = lexical_sub(old_src, wq)
    if new_src is not None:
        new_corpus.append(new_src)
    # Augmentation이 없더라도 원본 문장을 포함시킵니다
    new_corpus.append(old_src)

print(new_corpus[:10])

  0%|          | 0/594 [00:00<?, ?it/s]

['odor this.', 'smell this.', "did tom n't anything?", 'did tom do anything?', "would you prefer n't to help?", 'would you prefer not to help?', "i won't let nothing happen to you.", "i won't let anything happen to you.", 'tom says this now absurd.', 'tom says this is absurd.']


## 멋진 챗봇 만들기

In [ ]:
import numpy
import pandas
import torch
import nltk
import gensim

print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.0.2
2.2.2
2.10.0+cu128
3.9.1
4.4.0


In [ ]:
import pandas as pd

# 1. 데이터 로드
csv_path = 'ChatbotData.csv'
data = pd.read_csv(csv_path)

# 2. 질문과 답변 분리
questions = data['Q']
answers = data['A']

print(f"질문 개수: {len(questions)}")
print(f"답변 개수: {len(answers)}")

질문 개수: 11823
답변 개수: 11823


In [ ]:
import re

def preprocess_sentence(sentence):
    # 1. 영문자는 모두 소문자로 변환
    sentence = sentence.lower()

    # 2. 영문자, 한글, 숫자, 그리고 주요 특수문자(.,?!)를 제외한 문자는 공백으로 변환
    sentence = re.sub(r'[^a-z가-힣0-9.,?!]', ' ', sentence)

    # 3. 연속된 공백을 하나의 공백으로 줄이고 양쪽 공백 제거
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    return sentence

# 테스트 예시
sample_idx = 0
print(f"원본: {questions[sample_idx]}")
print(f"정제: {preprocess_sentence(questions[sample_idx])}")

원본: 12시 땡!
정제: 12시 땡!


### 🔧 튜터봇 수정사항: Step 3. 데이터 토큰화 (누락된 셀 복구)\n
이전 과정에서 `build_corpus` 함수 및 토큰화 로직이 누락되어 `que_corpus`가 정의되지 않는 에러가 발생할 수 있습니다. 이를 해결하기 위해 토큰화 코드를 다시 추가했습니다.

In [ ]:
from konlpy.tag import Mecab

mecab = Mecab()

def build_corpus(src_data, tgt_data, tokenize_fn, max_len=40):
    que_corpus = []
    ans_corpus = []

    seen_src = set()
    seen_tgt = set()

    for src, tgt in zip(src_data, tgt_data):
        clean_src = preprocess_sentence(src)
        clean_tgt = preprocess_sentence(tgt)

        if clean_src in seen_src or clean_tgt in seen_tgt:
            continue

        seen_src.add(clean_src)
        seen_tgt.add(clean_tgt)

        tokenized_src = tokenize_fn(clean_src)
        tokenized_tgt = tokenize_fn(clean_tgt)

        if len(tokenized_src) <= max_len and len(tokenized_tgt) <= max_len:
            que_corpus.append(tokenized_src)
            ans_corpus.append(tokenized_tgt)

    return que_corpus, ans_corpus

que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs, max_len=40)
print(f"구축된 que_corpus 크기: {len(que_corpus)}")
print(f"구축된 ans_corpus 크기: {len(ans_corpus)}")

### 🔧 튜터봇 수정사항: Step 4. Augmentation (Gensim 버전 호환성 패치)\n
Kyubyong님의 `ko.bin` 파일은 이전 버전(Gensim 3.x)에서 학습되어 현재 Colab의 최신 버전(4.x)과 충돌합니다. `Could not load model: 'Word2Vec' object has no attribute 'wv'` 에러를 막기 위해 아래 셀에서 **Gensim을 3.8.3 버전으로 다운그레이드**합니다.\n
\n
**🚨 매우 중요 🚨**\n
아래 `!pip install gensim==3.8.3` 셀을 실행하신 후, **반드시 상단 메뉴에서 런타임(커널)을 다시 시작**해 주셔야 합니다! (코랩의 경우: `런타임` -> `세션 다시 시작`)

In [ ]:
!pip install gensim==3.8.3

In [ ]:
import gensim
import random
from tqdm import tqdm
import os
import zipfile

# 1. Word2Vec 모델 경로 설정
w2v_path = '/content/ko.bin'
zip_path = '/content/ko.zip'

# 파일 관리
if os.path.exists(w2v_path) and os.path.getsize(w2v_path) < 1000000:
    os.remove(w2v_path)

if not os.path.exists(w2v_path):
    if not os.path.exists(zip_path):
        !pip install -q gdown
        !gdown --id 1G0e_aEaB2-J03V0n3l3k6Q -O /content/ko.zip
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
    except Exception as e: print(f"Extraction error: {e}")

try:
    print(f"Loading vectors from {w2v_path} with gensim {gensim.__version__}...")
    # Gensim 3.8.3 환경에서 정상 로드
    wv_model = gensim.models.Word2Vec.load(w2v_path)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Could not load model: {e}")
    wv_model = None

def lexical_sub(sentence, model):
    if model is None: return None
    res = []
    try:
        toks = sentence if isinstance(sentence, list) else sentence.split()

        # Gensim 3.x 버전에 맞는 단어 확인
        valid_toks = [t for t in toks if t in model.wv.vocab]
        if not valid_toks: return None

        _from = random.choice(valid_toks)
        _to = model.wv.most_similar(_from)[0][0]

        for tok in toks:
            res.append(_to if tok == _from else tok)
        return res
    except Exception: return None

# Augmentation logic
aug_que_corpus = []
aug_ans_corpus = []

if wv_model is not None:
    aug_que_corpus.extend(que_corpus)
    aug_ans_corpus.extend(ans_corpus)

    for q, a in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus), desc="Augmenting"):
        q_sub = lexical_sub(q, wv_model)
        aug_que_corpus.append(q_sub if q_sub else q)
        aug_ans_corpus.append(a)

        a_sub = lexical_sub(a, wv_model)
        aug_que_corpus.append(q)
        aug_ans_corpus.append(a_sub if a_sub else a)

    print(f"Final Augmented Dataset Size: {len(aug_que_corpus)}")
else:
    print(f"Word2Vec model is unavailable. Proceeding with original dataset (Size: {len(que_corpus)}).")
    aug_que_corpus = list(que_corpus)
    aug_ans_corpus = list(ans_corpus)

In [ ]:
import tensorflow as tf

# 1. 타겟 데이터(ans_corpus)에 <start>와 <end> 토큰 추가
aug_ans_corpus = [['<start>'] + ans + ['<end>'] for ans in aug_ans_corpus]

# 2. 단어 사전(Vocabulary) 구축
# 챗봇 데이터는 소스와 타겟이 같은 언어이므로 Embedding 층 공유를 위해 전체 데이터를 결합하여 하나의 단어장을 만듭니다.
total_data = aug_que_corpus + aug_ans_corpus

# Keras Tokenizer 객체 생성 (이미 토큰화가 되어 있으므로 filter 기능은 끕니다)
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='', lower=False)
tokenizer.fit_on_texts(total_data)

# 3. 데이터 벡터화 (정수 인코딩)
enc_train = tokenizer.texts_to_sequences(aug_que_corpus)
dec_train = tokenizer.texts_to_sequences(aug_ans_corpus)

# 4. 패딩 (Padding) 처리
# 시퀀스 길이를 일정하게 맞추기 위해 가장 긴 시퀀스 길이에 맞춰 패딩 추가 (post: 뒤에 0을 채움)
enc_train = tf.keras.preprocessing.sequence.pad_sequences(enc_train, padding='post')
dec_train = tf.keras.preprocessing.sequence.pad_sequences(dec_train, padding='post')

print(f"단어장(Vocabulary) 크기: {len(tokenizer.word_index) + 1}")
print(f"enc_train 텐서 형상: {enc_train.shape}")
print(f"dec_train 텐서 형상: {dec_train.shape}")

단어장(Vocabulary) 크기: 6265
enc_train 텐서 형상: (7737, 32)
dec_train 텐서 형상: (7737, 42)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

# 1. 하이퍼파라미터 (제출 예시 기준)
n_layers = 1
d_model = 368
n_heads = 8
d_ff = 1024
dropout = 0.2

# 2. 훈련 파라미터
warmup_steps = 1000
batch_size = 64
epochs = 10

# 단어장 크기 및 시퀀스 길이
vocab_size = len(tokenizer.word_index) + 1
src_vocab_size = vocab_size
tgt_vocab_size = vocab_size
pos_len = enc_train.shape[1]

# 3. 모델 인스턴스화
# (이전에 정의된 Transformer(nn.Module) 사용)
transformer = Transformer(
    n_layers=n_layers,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    pos_len=pos_len,
    dropout=dropout,
    shared_fc=True,
    shared_emb=True  # 챗봇은 동일 언어이므로 Embedding 공유
).to(device)

# 4. DataLoader 설정
enc_train_tensor = torch.tensor(enc_train, dtype=torch.long)
dec_train_tensor = torch.tensor(dec_train, dtype=torch.long)
dataset = TensorDataset(enc_train_tensor, dec_train_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 5. 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss(ignore_index=0) # Keras pad_sequences는 기본으로 0을 채움

class CustomSchedule(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=1000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer)

    def get_lr(self):
        step = max(1, self.last_epoch)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]

optimizer = torch.optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = CustomSchedule(optimizer, d_model=d_model, warmup_steps=warmup_steps)

# 6. 훈련 루프
print(f"[{device}] 디바이스에서 훈련을 시작합니다.")
transformer.train()
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (src, tgt) in enumerate(dataloader):
        src = src.to(device)
        tgt = tgt.to(device)

        # 교사 강요(Teacher Forcing)를 위한 타겟 입력 및 정답 분리
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        optimizer.zero_grad()

        # 모델 예측
        outputs = transformer(src, tgt_in)
        # 만약 반환값이 tuple(예: logits, enc_attns, ...) 이라면 첫 번째가 logits
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch: {epoch + 1:02d} | Loss: {total_loss/len(dataloader):.4f}")

# 7. 예문에 대한 답변 생성 (Inference)
def evaluate(sentence, model, tokenizer):
    model.eval()

    # 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)

    # 정수 인코딩 및 텐서화
    src_seq = tokenizer.texts_to_sequences([tokens])
    src_tensor = torch.tensor(src_seq, dtype=torch.long).to(device)

    # 디코더 초기 입력 설정 (<start>)
    start_idx = tokenizer.word_index.get('<start>')
    end_idx = tokenizer.word_index.get('<end>')

    tgt_seq = [start_idx]
    tgt_tensor = torch.tensor([tgt_seq], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(pos_len):
            outputs = model(src_tensor, tgt_tensor)
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs

            next_word = logits.argmax(dim=-1)[:, -1].item()
            tgt_seq.append(next_word)

            if next_word == end_idx:
                break

            tgt_tensor = torch.tensor([tgt_seq], dtype=torch.long).to(device)

    # 정수 시퀀스를 텍스트로 변환
    decoded_words = []
    for idx in tgt_seq:
        if idx == start_idx: continue
        if idx == end_idx: break
        word = tokenizer.index_word.get(idx, '')
        decoded_words.append(word)

    return ' '.join(decoded_words)

examples = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있다는 소리야."
]

print("\n
Translations")
for i, ex in enumerate(examples):
    translation = evaluate(ex, transformer, tokenizer)
    print(f"> {i+1}. {translation} <end>")


SyntaxError: unterminated string literal (detected at line 150) (2177792498.py, line 150)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

def calculate_bleu(reference, candidate):
    # reference: 정답 문장 토큰 리스트
    # candidate: 예측 문장 토큰 리스트
    smoothie = SmoothingFunction().method1
    # sentence_bleu는 여러 개의 reference를 지원하므로 리스트로 감싸서 전달
    return sentence_bleu([reference], candidate, smoothing_function=smoothie)

# 임의의 테스트 셋을 추출하여 BLEU Score 측정
test_size = 10
# Augmentation 되지 않은 원본 데이터 범위 내에서 샘플링
test_indices = np.random.choice(len(questions), test_size, replace=False)

bleu_scores = []

print("========== 성능 측정 (BLEU Score) ==========")
for idx in test_indices:
    original_q = questions[idx]

    # 정답 토큰 리스트 (형태소 분석이 완료된 ans_corpus 활용)
    ref_a = ans_corpus[idx]

    # 예측 생성 (evaluate 내부에서 다시 전처리 및 형태소 분석을 수행함)
    pred_a_str = evaluate(original_q, transformer, tokenizer)
    pred_a = pred_a_str.split() # 생성된 텍스트를 공백 기준으로 토큰화

    score = calculate_bleu(ref_a, pred_a)
    bleu_scores.append(score)

    print(f"Q: {original_q}")
    print(f"A(정답): {' '.join(ref_a)}")
    print(f"A(예측): {pred_a_str}")
    print(f"BLEU: {score:.4f}\n
")

print("============================================")
print(f"전체 평균 BLEU Score: {np.mean(bleu_scores):.4f}")


SyntaxError: unterminated f-string literal (detected at line 35) (2755854426.py, line 35)